# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL: [https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json](https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json).

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant --quiet

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

# Print dataset overview
meta = dataset.metadata
print(f"{meta.name}: {meta.description}\n")
print(f"Authors (@id): {[a['@id'] for a in getattr(meta, 'author', [])]}")
print(f"Dataset @id: {meta.id}")
print(f"Date published: {getattr(meta, 'datePublished', 'N/A')}")
print(f"License: {meta.license}")


## 2. Data Overview
Review available record sets, fields, and their IDs.

We use the Croissant metadata to enumerate record sets and their corresponding fields. Each is referenced by `@id`.

In [ ]:
# List record sets, fields and columns by their @id

record_sets = getattr(meta, 'recordSet', [])
if not record_sets:
    # Try to enumerate them from schema
    try:
        # Use internal schema structure
        from urllib.request import urlopen
        import json

        raw_meta = json.load(urlopen(croissant_url))
        def resolve_entities(key):
            return [obj for obj in raw_meta.get('@graph', []) if obj.get('@type') == key]
        cr_recordset = 'cr:RecordSet'
        cr_field = 'cr:Field'
        cr_column = 'cr:column'
        record_sets_data = [obj for obj in raw_meta.get('@graph', []) if obj.get('@type') == cr_recordset]
        print("Record Sets found:")
        for rs in record_sets_data:
            print(f"Record Set @id: {rs['@id']}")
            if 'field' in rs:
                print(f"  Fields:")
                for f in rs['field']:
                    fid = f['@id'] if isinstance(f, dict) else f
                    print(f"   - Field @id: {fid}")
            if 'column' in rs:
                print(f"  Columns:")
                for c in rs['column']:
                    cid = c['@id'] if isinstance(c, dict) else c
                    print(f"   - Column @id: {cid}")
            print()
        # save ids for next step
        record_set_ids = [rs['@id'] for rs in record_sets_data]
    except Exception as e:
        print("Failed to find record sets in Croissant schema.")
        print(e)
else:
    print("Record Sets found in metadata:")
    record_set_ids = []
    for rs in record_sets:
        rs_id = rs['@id'] if isinstance(rs, dict) else rs
        record_set_ids.append(rs_id)
        print(f"Record Set @id: {rs_id}")
        # Optionally print fields
        fields = rs.get('field', []) if isinstance(rs, dict) else []
        for f in fields:
            print(f"  Field @id: {f['@id'] if isinstance(f, dict) else f}")

## 3. Data Extraction
Load data from the main record sets into DataFrames, using the record set `@id`s from the overview.

In [ ]:
# The record_set_ids found above (typically only one main record set in survey datasets)
if not record_set_ids:
    raise RuntimeError("Record set IDs not found. Check previous cell output.")

dataframes = {}
for record_set_id in record_set_ids:
    print(f"Loading records from record set @id: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    if records:
        dataframes[record_set_id] = pd.DataFrame(records)
        print(f"  Columns (@id): {list(dataframes[record_set_id].columns)}")
        display(dataframes[record_set_id].head())
    else:
        print(f"  No records found for {record_set_id}.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

For this demonstration, we'll select fields via their `@id` (column names) for a survey score, filter values, normalize, and group by a demographic field.

In [ ]:
# Example: Filter on GAD-7 score (for demonstration)
main_rs_id = record_set_ids[0]
df = dataframes[main_rs_id]

# Croissant columns are referenced by their @id; check actual column name
score_fields = [col for col in df.columns if 'GAD' in col or 'gad' in col or 'phq' in col or 'PSQ' in col]
numeric_field_id = score_fields[0] if score_fields else df.select_dtypes(include='number').columns[0]

print(f"Using numeric field @id: {numeric_field_id}")

# Filter records based on GAD-7 score threshold, e.g., >10
threshold = 10
filtered_df = df[df[numeric_field_id] > threshold]
print(f"Filtered records with {numeric_field_id} > {threshold}:")
display(filtered_df.head())

# Normalize the numeric field
filtered_df[f'{numeric_field_id}_normalized'] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
print(f"Normalized {numeric_field_id} for filtered records:")
display(filtered_df[[numeric_field_id, f'{numeric_field_id}_normalized']].head())

# Grouped analysis: group by a key demographic field
group_fields = [col for col in df.columns if 'gender' in col.lower() or 'age' in col.lower() or 'education' in col.lower()]
group_field = group_fields[0] if group_fields else df.columns[0]

if group_field in filtered_df.columns:
    grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().reset_index()
    print(f"Grouped data by {group_field} (mean {numeric_field_id}):")
    display(grouped_df.head())

## 5. Visualization
Visualize distributions of the selected numeric fields and relationships to demographic attributes. All fields referenced by their `@id`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot histogram of normalized GAD-7 scores
plt.figure(figsize=(8, 4))
sns.histplot(filtered_df[f'{numeric_field_id}_normalized'], bins=15, kde=True)
plt.title(f"Distribution of Normalized {numeric_field_id} for records > {threshold}")
plt.xlabel(f"Normalized {numeric_field_id} (@id: {numeric_field_id})")
plt.show()

# Boxplot by demographic group
if group_field in filtered_df.columns:
    plt.figure(figsize=(8, 4))
    sns.boxplot(x=filtered_df[group_field], y=filtered_df[numeric_field_id])
    plt.title(f"{numeric_field_id} by {group_field} (@id)")
    plt.xlabel(f"{group_field} (@id)")
    plt.ylabel(f"{numeric_field_id} (@id)")
    plt.show()

## 6. Conclusion
This notebook demonstrated how to:
- Load a dataset defined by a Croissant schema using `mlcroissant`.
- Reference dataset entities (`recordSet`, fields, columns) using their `@id`.
- Perform exploratory analysis by filtering and normalizing numeric fields, and grouping by demographic attributes.
- Visualize data distributions and relationships.

The FAIR² Mental Health Survey Dataset enables reproducible, AI-ready exploration of mental health in Kenya, supporting further data science and public health applications.
